In [1]:
from typing import Sequence

import pandas as pd
from pandas._typing import Axis

from core import Config

config = Config()

MODES: list[str] = ["r2", "mae", "rmse"]
MODE: str = MODES[1]

IDENTIFIER = {
    "B": "Basis",
    "I": "Imputation",
    "H": "Hard Routing",
    "S": "Soft Blending",
    "All": "Global",
    "global": "Global",
    "10": "Energie (10)",
    "15": "Grundstoffe (15)",
    "20": "Industriegüter (20)",
    "25": "Nicht-Basiskonsumgüter (25)",
    "30": "Basiskonsumgüter(30)",
    "35": "Gesundheitswesen (35)",
    "40": "Finanzen (40)",
    "45": "Informationstechnologie (45)",
    "50": "Kommunikationsdienste (50)",
    "55": "Versorger (55)",
    "60": "Immobilien (60)",
}

ROW: Axis = 0
COLUMN: Axis = 1

global_basis_files: dict[tuple[str, str], str] = {
    #("B", "D14 Thr50"): "newResults/Basis/14_basis_thresh_50_metrics.csv",
    ("B", "LT D14 Thr50"): "newResults/Basis/14_basis_thresh_50_log_metrics.csv",
    ("B", "WT LT D14 Thr50"): "newResults/Basis/14_basis_thresh_50_win_log_metrics.csv",
    ("B", "WT LT D14 Thr10"): "newResults/Basis/14_basis_thresh_10_win_log_metrics.csv",
    ("B", "WT LT D2 Thr10"): "newResults/Basis/2_basis_thresh_10_win_log_metrics.csv",
    ("B", "WT LT D6 Thr10"): "newResults/Basis/6_basis_thresh_10_win_log_metrics.csv",
    ("B", "WT LT D6 Thr50"): "newResults/Basis/6_basis_thresh_50_win_log_metrics.csv",
    ("B", "WT LT D6 Thr10 O"): "newResults/Basis/6_basis_thresh_50_win_log_Ordered_metrics.csv",
    ("B", "WT LT D6 Thr50 O"): "newResults/Basis/6_basis_thresh_50_win_log_Ordered_metrics.csv",
    ("B", "WT LT D4 Thr50 O"): "newResults/Basis/4_basis_thresh_50_win_log_Ordered_metrics.csv",
    ("B", "WT LT D2 Thr50 O"): "newResults/Basis/2_basis_thresh_50_win_log_Ordered_metrics.csv",
    ("B", "WT LT D8 Thr50 O"): "newResults/Basis/8_basis_thresh_50_win_log_Ordered_metrics.csv",
    ("B", "WT LT D14 Thr50 O"): "newResults/Basis/14_basis_thresh_50_win_log_Ordered_sh42_metrics.csv",
    ("B", "WT LT D14 Thr10 O"): "newResults/Basis/14_basis_thresh_10_win_log_Ordered_sh42_metrics.csv",
    ("B", "WT LT D4 Thr10 O"): "newResults/Basis/4_basis_thresh_10_win_log_Ordered_sh42_metrics.csv",
}

global_imputation_files: dict[tuple[str, str], str] = {
    #("I, "D14 Thr50"): "newResults/Imputed/14_imputed_thresh_50_metrics.csv",
    ("I", "LT D14 Thr50"): "newResults/Imputed/14_imputed_thresh_50_log_metrics.csv",
    ("I", "WT LT D14 Thr50"): "newResults/Imputed/14_imputed_thresh_50_win_log_metrics.csv",
    ("I", "WT LT D14 Thr10"): "newResults/Imputed/14_imputed_thresh_10_win_log_metrics.csv",
    ("I", "WT LT D2 Thr10"): "newResults/Imputed/2_imputed_thresh_10_win_log_metrics.csv",
    ("I", "WT LT D6 Thr10"): "newResults/Imputed/6_imputed_thresh_10_win_log_metrics.csv",
    ("I", "WT LT D6 Thr50"): "newResults/Imputed/6_imputed_thresh_50_win_log_metrics.csv",
    ("I", "WT LT D6 Thr10 O"): "newResults/Imputed/6_imputed_thresh_10_win_log_Ordered_metrics.csv",
    ("I", "WT LT D6 Thr50 O"): "newResults/Imputed/6_imputed_thresh_50_win_log_Ordered_metrics.csv",
    ("I", "WT LT D4 Thr50 O"): "newResults/Imputed/4_imputed_thresh_50_win_log_Ordered_metrics.csv",
    ("I", "WT LT D2 Thr50 O"): "newResults/Imputed/2_imputed_thresh_50_win_log_Ordered_metrics.csv",
    ("I", "WT LT D8 Thr50 O"): "newResults/Imputed/8_imputed_thresh_50_win_log_Ordered_metrics.csv",
    ("I", "WT LT D14 Thr50 O"): "newResults/Imputed/14_imputed_thresh_50_win_log_Ordered_metrics.csv",
    ("I", "WT LT D14 Thr10 O"): "newResults/Imputed/14_imputed_thresh_10_win_log_Ordered_sh42_metrics.csv",
    ("I", "WT LT D4 Thr10 O"): "newResults/Imputed/4_imputed_thresh_10_win_log_Ordered_sh42_metrics.csv",
}

stacked_files: dict[tuple[str, str], str] = {
    ("Hard Routing",
     "WT LT D4 Thr50 O"): "newResults/Stacked/4_imputed_thresh_50_win_log_hard_routing_Ordered_metrics.csv",
    ("Soft Blending",
     "WT LT D4 Thr50 O"): "newResults/Stacked/4_imputed_thresh_50_win_log_soft_blending_Ordered_metrics.csv",
}

SECTOR_NUMBERS: Sequence[int] = range(10, 61, 5)


def load_sector_paths(config_key: str, path: str) -> dict[tuple[str, str], str]:
    sector_paths: dict[tuple[str, str], str] = {}
    for sector_number in SECTOR_NUMBERS:
        sector_path = path.replace("?", str(sector_number))
        sector_paths[(str(sector_number), config_key)] = sector_path
    return sector_paths


def read_metrics(file: str, col: str, metric: str = "mae", drop: str | None = None) -> pd.DataFrame:
    metrics: pd.DataFrame = pd.read_csv(config.results_dir / file, index_col=0)
    metrics_group: pd.DataFrame = metrics.groupby("sector").agg(
        {
            metric: ["mean"],
        }
    )
    metrics_group.columns = metrics_group.columns.get_level_values(1)
    metrics_group.rename(columns={"mean": col}, inplace=True)
    if drop is not None:
        metrics_group.drop(labels=drop, axis=ROW, inplace=True)
    return metrics_group


def concat_metrics(files: dict[tuple[str, str], str], metric: str = "mae", align: Axis = COLUMN,
                   drop: str | None = None) -> pd.DataFrame:
    frames: list[pd.DataFrame] = [read_metrics(file, col, metric, drop) for (row, col), file in files.items()]
    metrics: pd.DataFrame = pd.DataFrame()
    for frame in frames:
        metrics = pd.concat([metrics, frame], axis=align)
    metrics.index.name = "sector"
    return metrics


sector_files: list[dict[tuple[str, str], str]] = []
sector_files.append(load_sector_paths(
    "WT LT D4 Thr50 O",
    "newResults/Sector/4_imputed_thresh_50_win_log_sector_?_Ordered_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D14 Thr50",
    "newResults/Sector/14_imputed_thresh_50_log_sector_?_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D8 Thr50 O",
    "newResults/Sector/8_imputed_thresh_50_win_log_sector_?_Ordered_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D2 Thr50 O",
    "newResults/Sector/2_imputed_thresh_50_win_log_sector_?_Ordered_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D6 Thr50 O",
    "newResults/Sector/6_imputed_thresh_50_win_log_sector_?_Ordered_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D6 Thr10 O",
    "newResults/Sector/6_imputed_thresh_10_win_log_sector_?_Ordered_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D6 Thr50",
    "newResults/Sector/6_imputed_thresh_50_win_log_sector_?_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D2 Thr10",
    "newResults/Sector/2_imputed_thresh_10_win_log_sector_?_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D6 Thr10",
    "newResults/Sector/6_imputed_thresh_10_win_log_sector_?_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D14 Thr50 O",
    "newResults/Sector/14_imputed_thresh_50_win_log_sector_?_Ordered_sh42_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D4 Thr10 O",
    "newResults/Sector/4_imputed_thresh_10_win_log_sector_?_Ordered_sh42_metrics.csv"
))
sector_files.append(load_sector_paths(
    "LT D14 Thr50",
    "newResults/Sector/14_imputed_thresh_50_log_sector_?_sh42_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D14 Thr10",
    "newResults/Sector/14_imputed_thresh_10_win_log_sector_?_sh42_metrics.csv"
))
sector_files.append(load_sector_paths(
    "WT LT D14 Thr10 O",
    "newResults/Sector/14_imputed_thresh_10_win_log_sector_?_Ordered_sh42_metrics.csv"
))

global_basis_metrics_mae = concat_metrics(global_basis_files)
global_basis_metrics_r2 = concat_metrics(global_basis_files, metric="r2")
global_imputation_metrics_mae = concat_metrics(global_imputation_files)
global_imputation_metrics_r2 = concat_metrics(global_imputation_files, metric="r2")

col_names: list[str] = ["Hard Routing", "Soft Blending"]
stacked_metrics_mae: pd.DataFrame = concat_metrics(stacked_files, align=COLUMN)
idx = stacked_metrics_mae.columns.names
stacked_metrics_mae.columns = col_names
stacked_all_metric_mae: pd.DataFrame = stacked_metrics_mae.loc[stacked_metrics_mae.index == "All"]
stacked_all_metric_mae = stacked_all_metric_mae.transpose()
stacked_all_metric_mae.index = idx[0]

stacked_metrics_r2: pd.DataFrame = concat_metrics(stacked_files, metric="r2", align=COLUMN)
stacked_metrics_r2.columns = col_names
idx = stacked_metrics_r2.columns.names
stacked_all_metric_r2: pd.DataFrame = stacked_metrics_r2.loc[stacked_metrics_r2.index == "All"]
stacked_all_metric_r2 = stacked_all_metric_r2.transpose()
stacked_all_metric_r2 = idx[0]

idx = ["Basis", "Imputation"]
global_all_metric_mae: pd.DataFrame = pd.concat([
    global_basis_metrics_mae.loc[global_basis_metrics_mae.index == "All"],
    global_imputation_metrics_mae.loc[global_imputation_metrics_mae.index == "All"]
], axis=ROW)
global_all_metric_r2: pd.DataFrame = pd.concat([
    global_basis_metrics_r2.loc[global_basis_metrics_r2.index == "All"],
    global_imputation_metrics_r2.loc[global_imputation_metrics_r2.index == "All"]
], axis=ROW)
global_all_metric_mae.index = idx
global_all_metric_r2.index = idx

sector_metrics_mae: pd.DataFrame = pd.DataFrame()
for files in sector_files:
    metrics = concat_metrics(files, align=ROW, drop="All")
    sector_metrics_mae = pd.concat([sector_metrics_mae, metrics], axis=COLUMN)
sector_metrics_r2: pd.DataFrame = pd.DataFrame()
for files in sector_files:
    metrics = concat_metrics(files, metric="r2", align=ROW, drop="All")
    sector_metrics_r2 = pd.concat([sector_metrics_r2, metrics], axis=COLUMN)

In [2]:
sector_metrics_mae.to_csv(config.results_dir / "sector_metrics_mae.csv")

In [3]:
def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))


def rgb_to_typst(rgb):
    return f'rgb("#{rgb[0]:02x}{rgb[1]:02x}{rgb[2]:02x}")'


def blend(c1, c2, t):
    return tuple(round(c1[i] + (c2[i] - c1[i]) * t) for i in range(3))

#LOW_COLOR = hex_to_rgb("#f7f2fa")
LOW_COLOR = hex_to_rgb("#fff33b")
#HIGH_COLOR = hex_to_rgb("#8e5db7")
HIGH_COLOR = hex_to_rgb("#e93e3a")


def heat_color(value: float, vmin, vmax):
    """
    Maps value to a lilac color.
    low value → light lilac (good for MAE, bad for R²)
    high value → dark lilac
    Works with negatives as long as vmin/vmax span the full range.
    """
    low = LOW_COLOR
    high = HIGH_COLOR

    if pd.isna(value):
        return None

    if math.isclose(vmin, vmax):
        t = 0.5
    else:
        t = max(0.0, min(1.0, (value - vmin) / (vmax - vmin)))

    return rgb_to_typst(blend(low, high, t))


def fmt(x, digits=3):
    if pd.isna(x):
        return "—"
    return f"{x:.{digits}f}"


def make_cell(value, color):
    if pd.isna(value):
        return "[—]"
    return f'table.cell(fill: {color})[{value:.3f}]'


def sanitize_typst_text(text: str) -> str:
    return str(text).replace("[", r"\[").replace("]", r"\]")


# ── Color helpers ──────────────────────────────────────────────────────────────
def heat_color_all(value: float, vmin: float, vmax: float) -> str | None:
    if pd.isna(value):
        return None
    t = 0.5 if math.isclose(vmin, vmax) else max(0.0, min(1.0, (value - vmin) / (vmax - vmin)))
    return rgb_to_typst(blend(LOW_COLOR, HIGH_COLOR, t))


def sanitize(text: str) -> str:
    return str(text).replace("[", r"\[").replace("]", r"\]")


def make_cell_all(value, vmin: float, vmax: float) -> str:
    if pd.isna(value):
        return "[—]"
    color = heat_color_all(float(value), vmin, vmax)
    return f'table.cell(fill: {color})[{value:.3f}]'


In [4]:
import pandas as pd
import math

SECTOR_LABELS = {
    "10": "Energie (10)",
    "15": "Grundstoffe (15)",
    "20": "Industriegüter (20)",
    "25": "Nicht-Basiskonsumgüter (25)",
    "30": "Basiskonsumgüter(30)",
    "35": "Gesundheitswesen (35)",
    "40": "Finanzen (40)",
    "45": "Informationstechnologie (45)",
    "50": "Kommunikationsdienste (50)",
    "55": "Versorger (55)",
    "60": "Immobilien (60)",
    "All": "Global",
    "global": "Global",
}


def build_typst_table(df: pd.DataFrame) -> str:
    df = df.copy()
    model_cols = df.columns.tolist()

    values = df.stack().astype(float)
    vmin = values.min()
    vmax = values.max()

    lines = ['#table(']

    columns = ["1fr"] * len(model_cols)
    align = ["center + horizon"] * len(model_cols)
    lines.append(f'  columns: (3fr,{", ".join(columns)}),')
    lines.append('  inset: 4pt,')
    lines.append(f'  align: (left + horizon,{", ".join(align)}),')
    lines.append('  stroke: 0.4pt + rgb("#d8cde2"),')
    lines.append('')

    header_cells = ['[Modell]'] + [f'[{sanitize_typst_text(c)}]' for c in model_cols]
    lines.append(f'  table.header({", ".join(header_cells)}),')
    lines.append('')

    for sector_raw, row in df.iterrows():
        sector_label = SECTOR_LABELS.get(str(sector_raw), str(sector_raw))
        out_row = [f'[{sanitize_typst_text(sector_label)}]']

        for col in model_cols:
            val = row[col]
            color = heat_color(float(val), vmin, vmax) if pd.notna(val) else None
            out_row.append(make_cell(float(val), color) if pd.notna(val) else "[—]")

        lines.append("  " + ", ".join(out_row) + ",")

    lines.append(')')
    return "\n".join(lines)


if __name__ == "__main__":
    typst_code = build_typst_table(global_all_metric_r2)

In [5]:
import pandas as pd

# ── Labels ────────────────────────────────────────────────────────────────────
SECTOR_LABELS = {
    "10": "Energie (10)",
    "15": "Material: Roh- und Grundstoffe (15)",
    "20": "Industriegüter (20)",
    "25": "Nicht-Basiskonsumgüter (25)",
    "30": "Basiskonsumgüter (30)",
    "35": "Gesundheitswesen (35)",
    "40": "Finanzen (40)",
    "45": "Informationstechnologie (45)",
    "50": "Kommunikationsdienstleistungen (50)",
    "55": "Versorgungsunternehmen (55)",
    "60": "Immobilien (60)",
}

GLOBAL_LABELS = {
    "Basis": "Basis",
    "Imputation": "Imputation",
}

STACKING_LABELS = {
    "Hard Routing": "Hard Routing",
    "Soft Blending": "Soft Blending",
}


def build_big_table(
        df_global: pd.DataFrame,  # index = row label, cols = config names
        df_sector: pd.DataFrame,  # index = sector code (str), cols = config names
        df_stacking: pd.DataFrame,  # index = row label, cols = config names
) -> str:
    """
    Merges three panels into one Typst figure table.

    - Columns are the *union* of all config names in insertion order
      (global first, then sector, then stacking).
    - Missing configs in any panel render as [—].
    - Heatmap range is computed globally across all numeric cells.
    """
    # Unified column order
    all_cols: list[str] = []
    seen: set[str] = set()
    for df in (df_global, df_sector, df_stacking):
        for c in df.columns:
            if c not in seen:
                all_cols.append(c)
                seen.add(c)

    n_cols_total = len(all_cols) + 1  # +1 for row-label column

    # Global heatmap range
    all_values: pd.DataFrame = pd.concat([
        df_global.stack(future_stack=True).dropna(),
        df_sector.stack(future_stack=True).dropna(),
        df_stacking.stack(future_stack=True).dropna(),
    ]).astype(float)
    vmin: float = float(all_values.to_numpy().min())
    vmax: float = float(all_values.to_numpy().max())

    # Row helper
    def row_cells(label: str, row_data: pd.Series) -> str:
        cells = [f'[{sanitize(label)}]']
        for col in all_cols:
            val = row_data.get(col, float("nan"))
            cells.append(make_cell_all(val, vmin, vmax))
        return "    " + ", ".join(cells) + ","

    def panel_header(title: str) -> str:
        return f'    table.cell(colspan: {n_cols_total})[*{title}*],'

    # Typst column spec
    col_widths = ["3fr"] + ["1fr"] * len(all_cols)
    col_aligns = ["left + horizon"] + ["center + horizon"] * len(all_cols)
    header_cells = ["[Emissionswerte]"] + [f'[{sanitize(c)}]' for c in all_cols]

    lines = [
        "table(",
        f"  columns: ({', '.join(col_widths)}),",
        "  inset: 4pt,",
        f"  align: ({', '.join(col_aligns)}),",
        '  stroke: 0.4pt + rgb("#d8cde2"),',
        "",
        f"  table.header({', '.join(header_cells)},),",
        "",
        panel_header("Panel a: globale Modelle"),
    ]

    for idx, row in df_global.iterrows():
        lines.append(row_cells(GLOBAL_LABELS.get(str(idx), str(idx)), row))

    lines += ["", panel_header("Panel b: Sektor Modelle")]
    for idx, row in df_sector.iterrows():
        lines.append(row_cells(SECTOR_LABELS.get(str(idx), str(idx)), row))

    lines += ["", panel_header("Panel c: Stacking-Modelle")]
    for idx, row in df_stacking.iterrows():
        lines.append(row_cells(STACKING_LABELS.get(str(idx), str(idx)), row))

    lines += [
        "),",
    ]

    return "\n".join(lines)


if __name__ == "__main__":
    typst_code_all = build_big_table(global_all_metric_r2, sector_metrics_r2, stacked_all_metric_r2)


In [6]:
global_col: str = "WT LT D4 Thr50 O"
sector_col: str = global_col

global_sectors: pd.DataFrame = global_imputation_metrics_r2[[global_col]]
sectors: pd.DataFrame = sector_metrics_r2[["WT LT D4 Thr50 O"]]

global_sectors.rename(columns={global_col: f"Global {global_col}"}, inplace=True)
sectors.rename(columns={sector_col: f"Sectors {sector_col}"}, inplace=True)

sector_df: pd.DataFrame = pd.concat([
    global_sectors,
    sectors,
    stacked_metrics_r2,
], axis=COLUMN)

if __name__ == "__main__":
    typst_code_sector_vs = build_typst_table(sector_df)
    print("Done")

Done
